1.- intalacion de librerias

In [1]:
!pip install openai streamlit faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 67.6 MB/s eta 0:00:00


2.-Importaciones y conexión con Groq

In [2]:
from google.colab import userdata
from google.colab import files
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import os
import re


groq_api_key = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Cliente Groq inicializado")

Cliente Groq inicializado


3.- Probar conexión con Groq

In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Responde exactamente con estas dos palabras: conexión correcta"
        }
    ],
    temperature=0.1,
    max_tokens=100,
    reasoning_effort="low"
)

print(response.choices[0].message.content)

conexión correcta


4.- Subir y leer los PDF

In [4]:
archivos = files.upload()

documentos = []

for nombre_archivo in archivos.keys():

    if nombre_archivo.lower().endswith(".pdf"):

        lector = PdfReader(nombre_archivo)
        texto_completo = ""

        for pagina in lector.pages:
            texto = pagina.extract_text()

            if texto:
                texto_completo += texto + "\n"

        documentos.append({
            "nombre": nombre_archivo,
            "texto": texto_completo
        })

print("Documentos cargados:", len(documentos))

for documento in documentos:
    print("-", documento["nombre"])

Saving Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf to Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf
Saving NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf to NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf
Saving Reglamento_Titulacion_Duoc_searchable.pdf to Reglamento_Titulacion_Duoc_searchable.pdf
Documentos cargados: 3
- Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf
- NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf
- Reglamento_Titulacion_Duoc_searchable.pdf


5.- Revisar que los PDF se hayan leído bien

In [5]:
for documento in documentos:

    print("\n==============================")
    print("Documento:", documento["nombre"])
    print("==============================")

    print(documento["texto"][:1000])


Documento: Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf
Malla Curricular Carrera
Número de Currículum
Título que otorga
Modalidad
Salida Intermedia
Continuidad
:
:
:
:
:
:
Ingeniería en Informática mención Desarrollo de Software
1446116
Ingeniero(a) en Informática Especialización en Desarrollo de Software
Diurno Presencial
-
-
1° NIVEL 2° NIVEL 3° NIVEL 4° NIVEL 5° NIVEL 6° NIVEL 7° NIVEL 8° NIVEL
ESPECIALI-
DAD
FPY1101
FUNDAMENTOS DE PROGRAMACIÓN
DSY1102
DESARROLLO ORIENTADO A
OBJETOS
DSY1103
DESARROLLO FULL STACK I
DSY1105
DESARROLLO DE APLICACIONES
MÓVILES
GPY1101
EVALUACIÓN DE PROYECTOS DE
SOFTWARE
GPY1102
GESTION DE PROYECTOS DE
SOFTWARE
ISY1103
ARQUITECTURAS TI
CONTEMPORÁNEAS
TSY1101
TALLER APLICADO DE SOFTWARE
BIY1101
BASES DE INNOVACIÓN
BDY1101
BASE DE DATOS APLICADA I
BDY1102
BASE DE DATOS APLICADA II
DSY1104
DESARROLLO FULL STACK II
DSY1106
DESARROLLO FULL STACK III
DSY1107
DESARROLLO CLOUD NATIVE I
DSY1108
DESARROLL

6.- Crear los chunks

In [6]:
def chunking_text(text, chunk_size=200, overlap=50):

    words = text.split()
    chunks = []

    if overlap >= chunk_size:
        overlap = chunk_size - 1

    step = chunk_size - overlap

    for i in range(0, len(words), step):

        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

        if i + chunk_size >= len(words):
            break

    return chunks


chunks = []

for documento in documentos:

    partes = chunking_text(
        documento["texto"],
        chunk_size=200,
        overlap=50
    )

    for parte in partes:

        chunks.append({
            "texto": parte,
            "fuente": documento["nombre"]
        })

print("Total de chunks creados:", len(chunks))

Total de chunks creados: 92


7.- Agregar información estructurada de la malla

In [7]:
malla_estructurada = """
Malla curricular de Ingeniería en Informática mención Desarrollo de Software.

La asignatura PPY4616 Práctica Profesional se encuentra en el 8° nivel de la carrera.
"""

chunks.append({
    "texto": malla_estructurada,
    "fuente": "Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf"
})

print("Cantidad total de chunks:", len(chunks))

Cantidad total de chunks: 93


8.- Agregar fuente externa

In [36]:
# FUENTE EXTERNA

fuente_externa = """
Fuente externa: Ley 21.790 de Chile.

Las instituciones de educación superior deben establecer normas internas
para aplicar las disposiciones de esta ley.

La Superintendencia de Educación Superior es responsable de fiscalizar
su cumplimiento.
"""

chunks.append({
    "texto": fuente_externa,
    "fuente": "Ley 21.790 - Biblioteca del Congreso Nacional de Chile"
})

print("Fuente externa agregada")
print("Cantidad total de chunks:", len(chunks))

Fuente externa agregada
Cantidad total de chunks: 94


9.- Inicializar el modelo de embeddings

In [37]:
modelo_embeddings = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo de embeddings cargado")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo de embeddings cargado


10.- Generar los embeddings de todos los chunks

In [38]:
textos_chunks = [chunk["texto"] for chunk in chunks]

embeddings = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True
)

print("Cantidad de textos:", len(textos_chunks))
print("Forma de embeddings:", embeddings.shape)

Cantidad de textos: 94
Forma de embeddings: (94, 384)


11.- Crear el índice FAISS

In [39]:
dimension = embeddings.shape[1]

indice_faiss = faiss.IndexFlatL2(dimension)

indice_faiss.add(
    embeddings.astype("float32")
)

print("Vectores guardados en FAISS:", indice_faiss.ntotal)

Vectores guardados en FAISS: 94


12.- Crear la función de búsqueda

In [40]:
import re
import unicodedata
from sklearn.metrics.pairwise import cosine_similarity


def limpiar_texto(texto):

    texto = texto.lower()

    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        letra for letra in texto
        if unicodedata.category(letra) != "Mn"
    )

    texto = re.sub(r"[^a-z0-9\s]", " ", texto)

    palabras_comunes = {
        "el", "la", "los", "las",
        "un", "una", "unos", "unas",
        "de", "del", "al",
        "en", "que", "es", "esta",
        "este", "esta", "por", "para",
        "y", "o", "a"
    }

    palabras = texto.split()

    palabras = [
        palabra
        for palabra in palabras
        if palabra not in palabras_comunes
    ]

    return palabras


def buscar_chunks(pregunta, k=5):

    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True
    )

    similitudes = cosine_similarity(
        embedding_pregunta,
        embeddings
    )[0]

    palabras_pregunta = set(
        limpiar_texto(pregunta)
    )

    resultados = []

    for i, chunk in enumerate(chunks):

        palabras_chunk = set(
            limpiar_texto(chunk["texto"])
        )

        coincidencias = len(
            palabras_pregunta.intersection(palabras_chunk)
        )

        puntaje_palabras = coincidencias / max(
            len(palabras_pregunta),
            1
        )

        puntaje_final = (
            0.7 * similitudes[i]
            + 0.3 * puntaje_palabras
        )

        resultados.append({
            "texto": chunk["texto"],
            "fuente": chunk["fuente"],
            "similitud": similitudes[i],
            "coincidencias": coincidencias,
            "puntaje": puntaje_final
        })

    resultados.sort(
        key=lambda x: x["puntaje"],
        reverse=True
    )

    return resultados[:k]

13.- probar solamente la búsqueda

In [41]:
pregunta = "¿Qué pasa si repruebo una asignatura?"

resultados = buscar_chunks(pregunta, k=5)

for i, resultado in enumerate(resultados, 1):

    print("\n==============================")
    print("Resultado", i)
    print("Fuente:", resultado["fuente"])
    print("==============================")

    print(resultado["texto"])


Resultado 1
Fuente: NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf
inscribió la o las asignaturas señaladas. La calificación no podrá permanecer pendiente por más de un período académico. La Subdirección Académica de la Sede podrá, en el caso que un/a estudiante se vea imposibilitado, de realizar cualquiera de sus prácticas o internados por una causa grave e imprevista que no le fueran imputables, autorizar la extensión de la postergación de la calificación final de la asignatura por un periodo académico adicional. Mientras un/a estudiante no apruebe la o las asignaturas cuya calificación final haya sido postergada, no podrá inscribirse en aquellas asignaturas para las que éstas constituyen requisitos. Los plazos para solicitar nota/s pendiente/s, será establecido en el Calendario Académico del periodo. Sin perjuicio de lo dispuesto precedentemente, los y las estudiantes que cuenten con acreditación vigente, validada por Duoc UC, de encontrarse en alguna de las situaciones previstas en la Le

14.- Crear la función principal de EduRAG

In [13]:
def responder_pregunta(pregunta):

    resultados = buscar_chunks(pregunta, k=5)

    contexto = ""

    for i, resultado in enumerate(resultados, 1):

        contexto += f"""
Documento {i}
Fuente: {resultado["fuente"]}

{resultado["texto"]}

"""

    prompt = f"""
Eres un asistente académico para estudiantes de Duoc UC.

Debes responder únicamente utilizando la información
del contexto proporcionado.

Si la respuesta no aparece en el contexto, responde:
"No encontré información suficiente en los documentos disponibles."

Contexto:
{contexto}

Pregunta:
{pregunta}

Responde de forma clara, breve y fácil de entender.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=500,
        reasoning_effort="low"
    )

    print("Pregunta:")
    print(pregunta)

    print("\nRespuesta:")
    print(response.choices[0].message.content)

    print("\nFuentes utilizadas:")

    fuentes = set()

    for resultado in resultados:
        fuentes.add(resultado["fuente"])

    for fuente in fuentes:
        print("-", fuente)

15.- Preguntas de prueba

1

In [14]:
responder_pregunta(
    "¿Qué pasa si repruebo una asignatura?"
)

Pregunta:
¿Qué pasa si repruebo una asignatura?

Respuesta:
Si repruebas una asignatura:

- **Primera, segunda o tercera vez**: la reprobación queda registrada. Si la repruebas por tercera vez, serás eliminado/a académicamente, salvo excepciones que se indican en el artículo siguiente.

- **Cuarta vez**: solo puedes volver a cursarla con autorización previa de la Dirección de Carrera y cumpliendo al menos una de estas condiciones:  
  a) Tener un promedio ponderado ≥ 5,0.  
  b) Haber aprobado ≥ 80 % de los créditos totales del plan de estudios.  
  c) Obtener autorización excepcional de la Vicerrectoría Académica.

- **Si vuelves a reprobarla en la cuarta oportunidad**, serás eliminado/a del plan de estudios sin derecho a apelación y no podrás matricularte nuevamente en ningún plan activo de tu carrera.

En resumen, la reprobación repetida puede llevar a la eliminación académica y a la imposibilidad de continuar la carrera.

Fuentes utilizadas:
- NUEVO-REGLAMENTO-ACADEMICO_20260525.pd

2

In [15]:
responder_pregunta(
    "¿Qué requisitos necesito para realizar la práctica profesional?"
)

Pregunta:
¿Qué requisitos necesito para realizar la práctica profesional?

Respuesta:
**Requisitos para realizar la práctica profesional**

1. **Ser alumno regular** – Debes estar matriculado y con matrícula vigente en la carrera correspondiente.  
2. **Cumplir con el currículo** – La práctica se inscribe según el número de créditos que indique el plan de estudios de tu carrera.  
3. **Inscripción de la asignatura** – La asignatura de práctica (o internado) solo puede inscribirse cuando cumples el punto 1.  
4. **Flexibilización** – Si eres estudiante cuidador (Ley N° 21.790) o tienes alguna situación excepcional, puedes aplicar las normas de flexibilización académica, justificación de inasistencias o suspensión de estudios, siempre que no afecte el logro de los resultados de aprendizaje.  

En caso de que, por alguna razón, no puedas realizar la práctica en el período en que la inscribiste, puedes acogerte a lo dispuesto en el artículo 48 del Título XII (“Retiro y Postergación de Asig

3

In [26]:
responder_pregunta(
    "¿En qué nivel está la práctica profesional?"
)

Pregunta:
¿En qué nivel está la práctica profesional?

Respuesta:
La práctica profesional está ubicada en el **8° nivel** de la carrera.

Fuentes utilizadas:
- Malla-Curricular-1446116-2024-Ingenieroa-en-Informatica-Especializacion-en-Desarrollo-de-Software-1.pdf
- NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf
- Reglamento_Titulacion_Duoc_searchable.pdf


4

In [42]:
responder_pregunta(
    "¿Quién fiscaliza el cumplimiento de esta normativa en educación superior?"
)

Pregunta:
¿Quién fiscaliza el cumplimiento de esta normativa en educación superior?

Respuesta:
La Superintendencia de Educación Superior es la entidad responsable de fiscalizar el cumplimiento de la normativa en educación superior.

Fuentes utilizadas:
- Ley 21.790 - Biblioteca del Congreso Nacional de Chile
- NUEVO-REGLAMENTO-ACADEMICO_20260525.pdf
- Reglamento_Titulacion_Duoc_searchable.pdf


16.- Crear una función para evaluar fidelidad de la respuesta

In [32]:
def evaluar_fidelidad(pregunta, contexto, respuesta):

    prompt = f"""
Evalúa si la respuesta está basada únicamente en el contexto proporcionado.

Pregunta:
{pregunta}

Contexto:
{contexto}

Respuesta:
{respuesta}

Responde con un número entero del 1 al 10.

1 significa que la respuesta no está basada en el contexto.
10 significa que la respuesta está completamente basada en el contexto.

Responde solamente con el número.
"""

    evaluacion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=150,
        reasoning_effort="low"
    )

    return evaluacion.choices[0].message.content

16.1.- probar fidelidad

In [34]:
pregunta = "¿En qué nivel está la práctica profesional?"

contexto = """
La asignatura PPY4616 Práctica Profesional
se encuentra en el 8° nivel de la carrera.
"""

respuesta = "La práctica profesional está ubicada en el 8° nivel de la carrera."

nota_fidelidad = evaluar_fidelidad(
    pregunta,
    contexto,
    respuesta
)

print("Fidelidad:", nota_fidelidad)

Fidelidad: 10


17.- Crear una función para evaluar relevancia

In [33]:
def evaluar_relevancia(pregunta, respuesta):

    prompt = f"""
Evalúa qué tan bien la respuesta responde a la pregunta.

Pregunta:
{pregunta}

Respuesta:
{respuesta}

Responde con un número entero del 1 al 10.

1 significa que la respuesta no responde a la pregunta.
10 significa que la respuesta responde completamente a la pregunta.

Responde solamente con el número.
"""

    evaluacion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=150,
        reasoning_effort="low"
    )

    return evaluacion.choices[0].message.content

17.1 probar relevancia

In [35]:
pregunta = "¿En qué nivel está la práctica profesional?"

respuesta = "La práctica profesional está ubicada en el 8° nivel de la carrera."

nota_relevancia = evaluar_relevancia(
    pregunta,
    respuesta
)

print("Relevancia:", nota_relevancia)

Relevancia: 10
